In [7]:
import os
print(os.getcwd())
# use this working directory address to specify the path 
#add data.xlsx file in this directory

C:\Users\Abdul moyeed


In [3]:
import pandas as pd

# Load dataset
#please specify the data.xlsx file path 
df = pd.read_excel(r'C:\\Users\\Abdul moyeed\\Documents\data.xlsx', parse_dates=['time'])
df.set_index('time', inplace=True)


C:\Users\Abdul moyeed\AppData\Local\Programs\Python\Python313\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [4]:

import os, re, faiss, pdfplumber
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from transformers import pipeline



folder = "./docs"   # put a few PDFs here
docs_data = []

for file in os.listdir(folder):
    if file.lower().endswith(".pdf"):
        path = os.path.join(folder, file)
        with pdfplumber.open(path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                # clean
                text = re.sub(r'\s+', ' ', text).strip()
                if len(text) > 100:  # skip empty pages
                    docs_data.append({
                        "source": file,
                        "page": page_num,
                        "text": text
                    })

df = pd.DataFrame(docs_data)
print("Loaded pages:", len(df))

# ============================================================
# 4. Split text into chunks (~500 tokens)
# ============================================================
def chunk_text(text, size=500, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), size - overlap):
        chunk = " ".join(words[i:i+size])
        chunks.append(chunk)
    return chunks

chunked_records = []
for _, row in df.iterrows():
    chunks = chunk_text(row["text"])
    for i, c in enumerate(chunks):
        chunked_records.append({
            "source": row["source"],
            "page": row["page"],
            "chunk_id": i,
            "text": c
        })

df_chunks = pd.DataFrame(chunked_records)
print("Chunks created:", len(df_chunks))

# ============================================================
# 5. Create embeddings & build FAISS index
# ============================================================
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(df_chunks["text"].tolist(), normalize_embeddings=True, show_progress_bar=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(np.array(embeddings))

# ============================================================
# 6. Define RAG query function
# ============================================================
qa_model = pipeline("text2text-generation", model="google/flan-t5-small")

def rag_query(query, top_k=3):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q_emb), top_k)
    
    retrieved = []
    for j, i in enumerate(ids[0]):
        row = df_chunks.iloc[i]
        retrieved.append((row["source"], row["page"], row["text"], float(scores[0][j])))
    
    # assemble context
    context_text = "\n\n".join(
        [f"[{src} - p.{pg}]\n{text}" for src, pg, text, _ in retrieved]
    )
    
    prompt = f"Answer the question using only the provided excerpts.\n\n{context_text}\n\nQuestion: {query}\n\nAnswer concisely with citations in the format [source - page]."
    answer = qa_model(prompt, max_length=200, temperature=0.2)[0]["generated_text"]
    return answer, retrieved

# ============================================================
# 7. Example Query
# ============================================================
query = "what is a cyclone "
answer, sources = rag_query(query)

print("🧠 Question:", query)
print("\n💬 Answer:\n", answer)
print("\n📚 Sources:")
for s, p, t, sc in sources:
    print(f"- {s} (page {p}) | score={sc:.2f}")

# ============================================================
# 8. Simple Evaluation (Precision / Recall demo)
# ============================================================
# Suppose ground truth source for this query is "manual1.pdf"
ground_truth = {"manual1.pdf"}  # update this according to your doc
retrieved_sources = {s for s, _, _, _ in sources}

precision = len(ground_truth & retrieved_sources) / len(retrieved_sources)
recall = len(ground_truth & retrieved_sources) / len(ground_truth)

print(f"\n📊 Precision@{len(sources)}: {precision:.2f}")
print(f"📈 Recall@{len(sources)}: {recall:.2f}")





Loaded pages: 168
Chunks created: 189


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Device set to use cpu
Token indices sequence length is longer than the specified maximum sequence length for this model (1693 > 512). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Question: what is a cyclone 

💬 Answer:
 A cyclone is a mechanically aided collector that uses inertia to separate PM from the gas stream as it spirals through the cyclone. The collection efficiency of a cyclone improves as the number of revolutions made by the gas and the gas velocity increase. Ultimately, however, the overall performance depends on the particle size distribution of the incoming gas stream. Cyclones are generally used for collection of medium-sized and coarse particles. Cyclones have a relatively simple construction and generally have no moving parts. Simple cyclones consist of an inlet cylindrical section, conical section, outlet gas duct, outlet dust tube, and collection hopper; a cyclone uses an induced draft fan to move the gas stream through the device. They are sized to provide the maximum inlet velocity possible for high separation without excessive turbulence. Multiclones (or multicyclones) consist of multiple small-diameter tubes in parallel, each of which 